In [4]:
import getpass
import platform
import socket
import os
import datetime
import base64
import re

def get_user_info():
    username = getpass.getuser()
    computer_name = socket.gethostname()
    now = datetime.datetime.now()
    date = now.strftime("%Y-%m-%d")
    time = now.strftime("%H:%M:%S")
    return username, computer_name, date, time

def generate_user_info_html():
    username, computer_name, date, time = get_user_info()
    return f"""
    <div style="margin-bottom:20px; padding:10px;  border-left:6px solid #ffc107;">
        <div class="user-info">

            <h2>📌 <strong>Informations utilisateur : </h2>
            <ul>
                    <li>👤 <strong>User:</strong> {username}<br></li>
                    <li>💻 <strong>Machine:</strong> {computer_name}<br></li>
                    <li>⏰ <strong>Heure:</strong> {time}</li>
                    <li>🕒 <strong>Date:</strong> {date}<br></li>

            </ul>

        </div>
    </div>

    """


import xml.etree.ElementTree as ET


def format_action(action_text):
    if action_text.strip().startswith(("while", "for", "foreach", "do", "if")):
        # Remplacer certains motifs pour mettre sur plusieurs lignes
        # Par exemple remplacer chaque ')\t' (fin d'instruction suivie d'un tabulation) par ')\n    '
        formatted = action_text.replace("):", "):\n")
        formatted = formatted.replace(") ", ")\n")
        formatted = formatted.replace("\t", "\n    ")

        # Puis ajouter indentation 4 espaces à chaque ligne sauf la première
        lines = formatted.split('\n')
        for i in range(1, len(lines)):
            lines[i] = "    " + lines[i]
        formatted = '\n'.join(lines)

        return f'<pre class="action-pre" style="white-space: pre-wrap; font-family: monospace;">{formatted}</pre>'
    else:
        return action_text


# Load XML
tree = ET.parse("../data/JunitReport.xml")
root = tree.getroot()

# Count test results
count_passed = count_failed = count_error = 0
for testcase in root.findall(".//testcase"):
    status = testcase.get('status', 'UNKNOWN').upper()
    if status == 'PASSED':
        count_passed += 1
    elif status == 'FAILED':
        count_failed += 1
    elif status == 'ERROR':
        count_error += 1

# Déterminer le titre
if root.tag == 'testsuites':
    suite_name = root.attrib.get('name')
    titre = suite_name if suite_name else 'Campaign'
else:
    titre = 'Campaign'

image_dir = "C:/Users/hbakl/OneDrive/Desktop/Projet PFA/screenshots_base64"
from collections import defaultdict

image_map = defaultdict(list)

for filename in os.listdir(image_dir):
    if filename.endswith(".png"):
        base_name = re.sub(r"_\d+.*\.png$", "", filename).strip().lower()
        with open(os.path.join(image_dir, filename), "rb") as img_file:
            b64_img = base64.b64encode(img_file.read()).decode("utf-8")
            image_map[base_name].append(b64_img)



# Ajoute ici le calcul du temps d'exécution
if root.tag == 'testsuites':
    time_exec = root.attrib.get('time')
    if time_exec is None:
        time_exec = 0.0
        for suite in root.findall('.//testsuite'):
            t = suite.attrib.get('time', '0')
            try:
                time_exec += float(t)
            except:
                pass
        time_exec = str(time_exec)
else:
    time_exec = root.attrib.get('time', '0')

# Extraction robuste du timestamp
timestamp = root.attrib.get('timestamp', '')

if not timestamp:
    # Si le timestamp est vide, essayons de le prendre à l'intérieur des enfants
    for child in root:
        timestamp = child.attrib.get('timestamp', '')
        if timestamp:
            break

# Extraire uniquement la date (YYYY-MM-DD)
date_part = timestamp.split("T")[0] if "T" in timestamp else timestamp


# Conversion du temps d'exécution en heures, minutes et secondes
try:
    total_seconds = int(float(time_exec))
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    time_exec_formatted = f"{hours}h {minutes}m {seconds}s"
except:
    time_exec_formatted = time_exec + "s"  # fallback si erreur






# HTML header
html_content = f"""
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="utf-8"/>
<title>Rapport des tests - Échecs détaillés</title>
<style>
/* [Styles CSS inchangés pour design, cartes, filtres, badges...] */
body {{
    font-family: 'Arial', sans-serif;
    margin: 20px;
    font-size: 16px;
    background-color: #f8f9fa;
    color: #333;
}}
h1 {{ text-align: center; color: #2c3e50; margin-bottom: 30px;font-size: 32px;
    font-weight: bold; }}
.dashboard {{
    display: flex; gap: 20px; margin-bottom: 30px;
}}
.card {{
    flex: 1; padding: 20px; border-radius: 12px; color: black;
    font-size: 18px; text-align: center;
    box-shadow: 0 2px 10px rgba(0,0,0,0.1);
}}
.card:hover {{ transform: scale(1.03); }}

.filter {{
    margin-bottom: 30px;
    background: white;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #ddd;
    box-shadow: 0 1px 4px rgba(0,0,0,0.05);
}}
.filter label {{ font-weight: 600; margin-right: 10px; }}

.testsuite-block {{
    margin-bottom: 40px;
    padding: 15px;
    border: 2px dashed #ccc;
    border-radius: 10px;
    background-color: #fefefe;
}}

.section {{
    margin-bottom: 30px;
    padding: 20px;
    border: 1px solid #e3e3e3;
    border-radius: 10px;
    background: white;
    box-shadow: 0 1px 5px rgba(0,0,0,0.05);
}}
.section h3 {{ margin-top: 0;
    font-style: italic;
    color: #007bff;
}}

table {{
    border-collapse: collapse;
    width: 100%;
    margin-top: 10px;
    margin-bottom: 10px;
}}
td {{
    font-size: 14px;
    font-family: monospace;
    color: #222;
}}

th, td {{
    border: 1px solid #dee2e6;
    padding: 10px;
    text-align: left;
    vertical-align: top;
}}
th {{
    background-color: #f1f1f1;
    font-weight: 600;
}}

.passed {{ background-color: #e6ffe6; }}
.failed {{ background-color: #ffe6e6; }}
.error {{ background-color: #fff3cd; }}

.badge {{
    display: inline-block;
    padding: 4px 10px;
    border-radius: 12px;
    font-size: 0.85em;
    font-weight: 600;
    color: white;
}}
.badge.passed {{ background-color: #28a745; }}
.badge.failed {{ font-weight: bold;
    color: white;
    background-color: red; }}
.badge.error {{ background-color: #ffc107; color: black; }}

.toggle-details {{
    background-color: #007bff;
    color: white;
    border: none;
    padding: 6px 12px;
    border-radius: 5px;
    cursor: pointer;
    margin: 8px 0;
}}

pre {{
    display: none;
    white-space: pre-wrap;
    font-size: 13px;
    background-color: #f6f8fa;
    border-left: 4px solid #ccc;
    padding: 10px;
    margin-top: 5px;
}}

/* Afficher les pre avec class action-pre dans les actions */
pre.action-pre {{
    display: block !important;
    background-color: #f6f8fa;
    border-left: 4px solid #ccc;
    padding: 10px;
    margin-top: 5px;
    white-space: pre-wrap;
    font-family: monospace;
    font-size: 13px;
}}

.step-wrapper, .testcases-wrapper {{
    display: none;
    margin-top: 10px;
}}

</style>
<script>
function filterTests() {{
    var selected = document.getElementById("statusFilter").value;
    var sections = document.querySelectorAll(".section");
    sections.forEach(function(section) {{
        var status = section.getAttribute("data-status");
        section.style.display = (selected === "all" || status === selected) ? "block" : "none";
    }});
}}
function toggleDetails(button) {{
    var preId = button.getAttribute("data-target");
    var pre = document.getElementById(preId);
    if (pre) {{
        pre.style.display = pre.style.display === "block" ? "none" : "block";
    }}
}}
function toggleSteps(button) {{
    var stepWrapper = button.nextElementSibling;
    stepWrapper.style.display = stepWrapper.style.display === "none" ? "block" : "none";
    button.textContent = stepWrapper.style.display === "none" ? "Afficher les étapes" : "Masquer les étapes";
}}
function toggleTestCases(button) {{
    var wrapper = button.nextElementSibling;
    wrapper.style.display = wrapper.style.display === "none" ? "block" : "none";
    button.textContent = wrapper.style.display === "none" ? "Afficher les cas de test" : "Masquer les cas de test";
}}
function toggleAllSuites() {{
    var blocks = document.querySelectorAll(".testsuite-block");
    blocks.forEach(block => {{
        block.style.display = block.style.display === "none" ? "block" : "none";
    }});
}}
</script>
</head>

<body>
<h1> {titre}</h1>
 {generate_user_info_html()}
<div class="dashboard">
    <div class="card passed-box">✅ <strong>Passed: {count_passed}</strong></div>
    <div class="card failed-box">❌ <strong>Failed: {count_failed}</strong></div>
    <div class="card time-box">⏱️ <strong>Temps d'exécution :</strong> {time_exec_formatted} secondes</div>
    <div class="card time-box">📅 <strong>Date :</strong> {date_part}</div>
</div>


<div class="filter">
    <label for="statusFilter"><strong>🎯 Filtrer par statut :</strong></label>
    <select id="statusFilter" onchange="filterTests()">
        <option value="all">Tous</option>
        <option value="passed">Passed ✅</option>
        <option value="failed">Failed ❌</option>
    </select>
</div>

"""

# Suites et testcases
suites = root.findall(".//testsuite") if root.tag == "testsuites" else [root]
# Code base64 de l'image
import base64
img_html = f'<img src="data:image/png;base64,{b64_img}" />'
for suite in suites:
    suite_name = suite.get("name", "TestSuite").strip() or "TestSuite"
    html_content += f"""
    <div class="testsuite-block">
        <h2>🧪 Test Suite : {suite_name}</h2>
        <button class="toggle-details" onclick="toggleTestCases(this)">Afficher les cas de test</button>
        <div class="testcases-wrapper">
    """

for testcase in suite.findall(".//testcase"):
    name = testcase.get('name', 'Unnamed Test')
    status = testcase.get('status', 'UNKNOWN').upper()
    time = testcase.get('time', '0')
    status_class = status.lower()

    html_content += f"""
    <div class="section" data-status="{status_class}">
        <h3>Test Case : {name}</h3>
        <p><strong>Statut :</strong> <span class="badge {status_class}">{status}</span></p>
        <p><strong>Durée :</strong> {time} secondes</p>
        <button class="toggle-details" onclick="toggleSteps(this)">Afficher les étapes</button>
        <div class="step-wrapper">
            <table>
                <thead>
                    <tr>
                        <th>Étape</th>
                        <th>Action</th>
                        <th>Durée (s)</th>
                        <th>Statut</th>
                        <th>Détails</th>
                    </tr>
                </thead>
                <tbody>
    """

    for idx, step in enumerate(testcase.findall(".//step"), 1):
        desc = step.get('description', f'Étape {idx}')
        action = step.get('action', '')
        step_time = step.get('time', '')
        step_status = step.get('status', '').upper()

        failure = step.find('failure')
        if failure is not None:
            message = failure.get('message', '—').strip()
            details = failure.text.strip() if failure.text else ''
            combined_details = f"💬 Message d'erreur :\n{message}\n\n📜 Détails :\n{details}"
            pre_id = f"details_{name.replace(' ', '_')}_{idx}"
            toggle_button = f"""<button class="toggle-details" onclick="toggleDetails(this)" data-target="{pre_id}">Afficher/Masquer détails</button>"""
            pre_block = f"""<tr><td colspan="5"><pre id="{pre_id}">{combined_details}</pre></td></tr>"""
        else:
            toggle_button = "—"
            pre_block = ""

        action_formatted = format_action(action)

        html_content += f"""
        <tr class="{step_status.lower()}">
            <td>{desc}</td>
            <td>{action_formatted}</td>
            <td>{step_time}</td>
            <td>{step_status}</td>
            <td>{toggle_button}</td>
        </tr>
        {pre_block}
        """

    html_content += """
                </tbody>
            </table>
    """

    # ⬇️ Insertion correcte de l’image associée au test case ici
    test_case_key = name.strip().lower()
    img_block = ""
    for key, b64_list in image_map.items():
        if key in test_case_key:
            for b64_img in b64_list:
                img_block += f"""
                <div class="image-wrapper" style="text-align: center;
                    margin-top: 20px;">
                    <img src="data:image/png;base64,{b64_img}"
                        style="max-width: 80%; height: auto;
                            border: 1px solid #ccc;
                            border-radius: 8px;
                            box-shadow: 2px 2px 10px rgba(0,0,0,0.1);" />
            </div>
            """


    html_content += img_block  # ⬅️ Juste ici
    html_content += """
        </div> <!-- end of step-wrapper -->
    </div> <!-- end of section -->
    """

# Sauvegarder le fichier
with open("../reports/rapport_tests_avec_img_dyn.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print("✅ Rapport HTML généré avec titre dynamique selon test")

✅ Rapport HTML généré avec titre dynamique selon test
